# MovieLens 100K — five recommenders from scratch

A guided walkthrough of the study. Run `python run_experiments.py` first so that
`results/` and `figures/` are populated; this notebook reads those artifacts and
re-derives a few of the headline numbers live, so you can see the models work rather
than only read their output.

**Contents**
1. Data and the sparsity problem
2. The long tail and the popularity groups every later analysis turns on
3. Leakage-free splitting and the roll-number target user
4. Fitting each paradigm and inspecting its internals
5. The accuracy / coverage trade-off
6. Explaining a single recommendation five different ways


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.pipeline import get_context, get_target, build_model, load_json, load_table

ds, split, ctx = get_context()
print(f'{ds.n_users} users x {ds.n_items} movies, {len(ds.ratings):,} ratings')


## 1. Data and sparsity

The rating matrix `R` is stored as a sparse CSR matrix. Missing ratings are
*structurally absent*, never dense zeros — that is what keeps a 'missing' from ever
being mistaken for a 'rated 0' downstream.


In [ ]:
from src.data import build_matrix, sparsity_stats
R = build_matrix(ds.ratings, ds.n_users, ds.n_items)
pd.Series(sparsity_stats(R))


## 2. The long tail

Head / medium / long tail are cut at 1/3 and 2/3 of *cumulative interaction mass*,
not at an arbitrary rating count. Each group therefore absorbs the same share of
observed attention.


In [ ]:
pg = ctx.pop_groups
print(f"head   : {pg['n_head']:5d} movies  (>= {pg['head_min_count']} train ratings)")
print(f"medium : {pg['n_medium']:5d} movies  (>= {pg['medium_min_count']})")
print(f"tail   : {pg['n_long_tail']:5d} movies")

lt = load_table('long_tail')
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(lt['rank'], 100*lt.cum_share)
ax.axvline(pg['n_head'], ls='--', c='crimson')
ax.axvline(pg['n_head']+pg['n_medium'], ls='--', c='darkorange')
ax.set_xlabel('movie rank by popularity'); ax.set_ylabel('cumulative % of interactions')
ax.set_title('Long-tail curve'); plt.show()


## 3. Split and target user

Per-user stratified 70/10/20. Hyper-parameters are chosen on validation; the test
split is read exactly once, in stage 08.


In [ ]:
print(split.summary())
tgt = get_target()
print(tgt.mapping_explanation)
print(f'visible profile: {len(tgt.kept)} ratings, hidden: {len(tgt.hidden)}')


## 4. The models

Each paradigm is refitted here on `R_train` using the configuration that stage 03-07
selected on validation. Fitting all five takes well under a minute.


In [ ]:
specs = {k: (load_json(k)['selection']['ranking_selected']
             if 'selection' in load_json(k) else load_json(k)['ranking_selected'])
         for k in ['ubcf','ibcf','regression','slim','graph']}
for k, s in specs.items():
    print(f'{k:11s} {s}')


In [ ]:
models = {k: build_model(s).fit(split.R_train) for k, s in specs.items()}
for k, m in models.items():
    print(f'{m.name:34s} fit {m.fit_time_s:6.2f}s  model {m.model_bytes()/1e6:6.2f} MB')


### SLIM's learned matrix

`W` is non-negative with a zero diagonal, and most of it is exactly zero. The
surviving coefficients are the item-item relationships the model decided were worth
paying for.


In [ ]:
slim = models['slim']
print(pd.Series(slim.sparsity()))
print()
for a, b, w in slim.strongest_pairs(8):
    print(f'{w:7.4f}   {ds.titles[a][:38]:40s} -> {ds.titles[b][:38]}')


### Similarity vs learned regression coefficient

The regression model keeps the same neighbourhoods as item-based CF but *learns* the
weights. Similarity is a marginal association; the ridge coefficient is a partial one.


In [ ]:
reg = models['regression']
sims, coefs, _ = reg.similarity_vs_coefficient()
from scipy.stats import pearsonr, spearmanr
print(f'Pearson  r = {pearsonr(sims, coefs)[0]:+.3f}')
print(f'Spearman rho = {spearmanr(sims, coefs)[0]:+.3f}')
print(f'negative coefficients: {100*(coefs<0).mean():.1f}%')

fig, ax = plt.subplots(figsize=(5.5,4.5))
ax.scatter(sims, coefs, s=2, alpha=0.12)
ax.axhline(0, c='grey', lw=.8); ax.set_ylim(-np.percentile(abs(coefs),99.5), np.percentile(abs(coefs),99.5))
ax.set_xlabel('adjusted-cosine similarity'); ax.set_ylabel('learned ridge coefficient')
plt.show()


## 5. The accuracy / coverage trade-off

This is the central result of the study.


In [ ]:
mu = load_table('multiuser_test')
pb = load_table('popularity_bias')
view = mu[mu.label.isin(['UBCF','IBCF','RegressionCF','SLIM','GraphRec','Popularity'])]
view[['label','rmse','ndcg_mean','recall_mean','catalog_coverage',
      'novelty_mean','ild_mean','tail_frac_mean','train_time_s']].round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(6,4.5))
for _, r in pb.iterrows():
    ax.scatter(100*r.catalog_coverage, r.ndcg_mean, s=70)
    ax.annotate(r.model, (100*r.catalog_coverage, r.ndcg_mean), fontsize=8,
                xytext=(4,4), textcoords='offset points')
ax.set_xlabel('catalog coverage (%)'); ax.set_ylabel('NDCG@10')
ax.set_title('Accuracy vs reach'); plt.show()


## 6. One recommendation, five explanations

Every model can justify its own output in the terms it actually used.


In [ ]:
from src.base import topk_from_scores
u = tgt.u
prof = split.train[split.train.u==u].sort_values('rating', ascending=False)
print('top-rated movies in the visible profile:')
for _, r in prof.head(5).iterrows():
    print(f'  {r.rating:.0f}  {ds.titles[int(r.i)]}')

for name, m in models.items():
    top = topk_from_scores(m.score_all(), split.R_train, 3)[u]
    print(f'\n--- {name} ---')
    for rank, i in enumerate(top, 1):
        print(f'  {rank}. {ds.titles[int(i)]}')
        for e in (m.explain(u, int(i), 2) or [])[:2]:
            print(f'       {e}')


---

The full write-up, with every table and all seventeen figures, is in
[`REPORT.md`](../REPORT.md); the portfolio pack is in [`PORTFOLIO.md`](../PORTFOLIO.md).
